In [12]:
import os
import glob
import tgt
import pandas as pd
import pickle
import tqdm
import warnings
warnings.simplefilter("ignore")


import sys
sys.path.insert(0, "../../..")

from utils import data_processing

In [13]:
with open(
    "../../pronunciation_counts/pronunciation_dict_fr.pkl",
    "rb"
) as f:
    pronunciation_dict = pickle.load(f)

In [14]:
DATASET_PATH = "/Volumes/Seagate Backup Plus Drive/ultrasound_llm/dataset/"
identifiers = []
for f in tqdm.tqdm(glob.glob(os.path.join(DATASET_PATH,"dataset/*_frames.npy"))):
    if os.path.isfile(f.replace("_frames", "_fps")) and os.path.isfile(
        f.replace("_frames.npy", ".TextGrid")
    ):
        identifier = os.path.basename(f).replace("_frames.npy", "")
        if "aud" in identifier or "spo" in identifier:
            identifiers += [identifier]


identifiers = [i for i in identifiers if i[3]=="e"]
tgrds = [tgt.read_textgrid(DATASET_PATH+"dataset/"+i+".TextGrid") for i in tqdm.tqdm(identifiers)]
art_fs = [DATASET_PATH+"dataset/"+i+"_frames.npy" for i in tqdm.tqdm(identifiers)]
ac_fs = [DATASET_PATH+"dataset_audio/"+i+"_audio.npy" for i in tqdm.tqdm(identifiers)]

100%|██████████| 7152/7152 [00:00<00:00, 6831624.28it/s]


In [ ]:
dfs_final = []

for tgrd, art_f, ac_f in tqdm.tqdm(zip(tgrds, art_fs, ac_fs), total=len(tgrds)):
    
    df_final = data_processing.alignment_to_dataframe(
        tgrd,
        lang="be",
        pronunciation_dict=pronunciation_dict
    )
    df_final["ac_f"] = ac_f
    df_final["art_f"] = art_f
    dfs_final += [df_final]

100%|██████████| 7152/7152 [00:07<00:00, 898.54it/s] 


In [19]:
df = pd.concat(dfs_final, ignore_index=True)

In [20]:
df.to_csv("datasets/processed_beus.csv", index=False)